# ISS Group 24 Modeling

**Before running:**
1. Open the [shared Drive folder](https://drive.google.com/drive/folders/19el_ISdRf5WoXarBsUOXjxFY9A4cPGYJ?usp=sharing) and click **"Add shortcut to Drive"** (anywhere in your Drive is fine).
2. Select a GPU runtime: *Runtime → Change runtime type → T4 GPU*.
3. Run cells **in order** top-to-bottom. Each stage cell is idempotent — re-running a completed stage skips it automatically via checkpoint detection.

In [1]:
import os
import sys
import subprocess
from pathlib import Path


In [2]:
SHARED_FOLDER_LINK = "https://drive.google.com/drive/folders/19el_ISdRf5WoXarBsUOXjxFY9A4cPGYJ?usp=sharing"
REPO_URL        = "https://github.com/pewpewnor/iss_group_24.git"
REPO_LOCAL_PATH = "/content/iss_group_24_src"


In [ ]:
from google.colab import drive


def mount_drive() -> str:
    """Mount Google Drive and return the iss_group_24 project root path."""
    drive.mount("/content/drive")
    for candidate in [
        "/content/drive/Shareddrives/iss_group_24",
        "/content/drive/MyDrive/iss_group_24",
    ]:
        if Path(candidate).exists():
            print(f"Drive mounted. Project root: {candidate}")
            return candidate
    raise RuntimeError(
        "Shared folder 'iss_group_24' not found on this Drive.\n"
        f"Open the link and add a shortcut to your Drive: {SHARED_FOLDER_LINK}"
    )

DRIVE_ROOT = mount_drive()

In [3]:
def setup_repo() -> None:
    """Clone the repo at depth=1, or reset to origin/main if already cloned."""
    if Path(REPO_LOCAL_PATH).exists():
        subprocess.run(["git", "-C", REPO_LOCAL_PATH, "fetch", "origin"], check=True)
        subprocess.run(
            ["git", "-C", REPO_LOCAL_PATH, "reset", "--hard", "origin/main"],
            check=True,
        )
        print(f"Repo updated to origin/main: {REPO_LOCAL_PATH}")
    else:
        subprocess.run(
            ["git", "clone", "--depth=1", REPO_URL, REPO_LOCAL_PATH],
            check=True,
        )
        print(f"Repo cloned: {REPO_LOCAL_PATH}")
    sys.path.insert(0, REPO_LOCAL_PATH)


def install_deps() -> None:
    """Install packages not pre-installed on Colab (torch/torchvision are pre-installed)."""
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "pillow>=10.0", "scipy", "matplotlib>=3.7",
        ],
        check=True,
    )
    print("Dependencies ready")


In [4]:
DRIVE_ROOT = "."
MANIFEST  = f"{DRIVE_ROOT}/dataset/cleaned/manifest.json"
DATA_ROOT = f"{DRIVE_ROOT}/dataset/cleaned"
MODEL_DIR = f"{DRIVE_ROOT}/model"

# Set CWD to the Drive project root so relative paths written by
# train.py (analysis/, manifest parent, etc.) land on the Drive.
os.chdir(DRIVE_ROOT)
print(f"Working directory: {os.getcwd()}")

Working directory: /mnt/Onetera/iss_group_24


In [ ]:
setup_repo()

In [ ]:
install_deps()

---
## Step 0 — Build Dataset Manifest (80/20 train/test split)

Copies and indexes images from `dataset/original/` into `dataset/cleaned/`.
Manifest carries only `train` and `test` splits; the trainer derives K-fold CV from train.
**Run once.** Skip if `manifest.json` already exists and the staged files are present.


In [6]:
import aggregator
from pathlib import Path

train_dir = Path(DATA_ROOT) / 'train'
test_dir = Path(DATA_ROOT) / 'test'
staged_ok = (
    Path(MANIFEST).exists()
    and train_dir.is_dir()
    and any(train_dir.iterdir())
    and test_dir.is_dir()
    and any(test_dir.iterdir())
)
if staged_ok:
    print(f'Dataset already staged at {DATA_ROOT} — skipping aggregator.')
else:
    if Path(MANIFEST).exists():
        print('Manifest found but staged files are missing — re-running aggregator.')
    else:
        print('Running aggregator (copies images to dataset/cleaned/) — may take several minutes')
    aggregator.main()


Dataset already staged at ./dataset/cleaned — skipping aggregator.


---
## Resume helper

`resume_from(name_or_path)` resolves a checkpoint reference for the `resume` kwarg of `train()`.

Recognised filenames (resolved against `MODEL_DIR`):
- `last.pt` (default — set `resume=True` for the same effect)
- `best.pt`
- `stage1_complete.pt`, `stage2_complete.pt`, `stage3_complete.pt`
- `ckpt_s1_epoch{E:03d}.pt`
- `ckpt_s2_epoch{E:03d}.pt`
- `ckpt_s3_epoch{E:03d}_fold{F}.pt`

Absolute paths are passed through as-is.


In [ ]:
from modeling.train import train
from modeling.evaluate import run as evaluate_run
from modeling.plot import plot_all_from_jsons

ANALYSIS_DIR = f'{DRIVE_ROOT}/analysis'

def resume_from(name_or_path: str | None):
    """Build the resume argument for train(). None -> True (auto from last.pt)."""
    if name_or_path is None:
        return True
    if Path(name_or_path).is_absolute():
        return str(name_or_path)
    return f'{MODEL_DIR}/{name_or_path}'


---
## Training

Three stages, every `(stage, epoch, fold)` checkpoint is resumable:

| Stage | Description | CV | Val source |
|---|---|---|---|
| **1** | Head warmup (backbone frozen) | no | sample of `test` split |
| **2** | Partial unfreeze (`features[7:]`) | no | sample of `test` split |
| **3** | Full unfreeze + K-fold rotating CV | yes (default K=3) | per-fold val partition |

Inside each Stage 3 epoch, every fold runs sequentially: the same model rotates through K folds.
Per-(epoch, fold) JSON is written to `analysis/stage{1,2,3}/`. No PNGs during training — call `plot_all_from_jsons` at the end.

**Stage-completion checkpoints** (`stageN_complete.pt`) are written once at the end of each stage and never auto-deleted. Pointing `resume` at one of these starts the next stage from a fresh optimizer.


In [ ]:
# All training hyperparameters live here. Edit any to override defaults.
TRAIN_KWARGS = dict(
    manifest         = MANIFEST,
    data_root        = DATA_ROOT,
    out_dir          = MODEL_DIR,
    analysis_dir     = ANALYSIS_DIR,

    # Stage durations (sized for ~12-14h on a Colab T4 with default episode counts)
    stage1_epochs    = 5,
    stage2_epochs    = 8,
    stage3_epochs    = 35,

    # K-fold CV (Stage 3 only)
    folds            = 3,
    fold_seed        = 42,

    # Episodes / batches
    episodes_per_epoch_s1 = 1500,
    episodes_per_epoch_s2 = 1500,
    episodes_per_epoch_s3 = 2000,
    val_episodes_s1  = 400,
    val_episodes_s2  = 400,
    val_episodes_s3  = 240,
    batch_size       = 16,
    num_workers      = 2,
    n_support        = 4,

    # Learning rates per stage
    lr_heads_s1            = 3e-4,
    lr_heads_s2            = 2e-4,
    lr_backbone_upper_s2   = 5e-5,
    lr_heads_s3            = 1e-4,
    lr_backbone_upper_s3   = 5e-6,
    lr_backbone_lower_s3   = 5e-6,
    weight_decay           = 1e-4,
    grad_clip              = 1.0,
    warmup_frac            = 0.05,

    # Loss / regularisation weights
    presence_weight     = 1.0,
    attn_loss_weight    = 0.5,
    aux_weight          = 0.5,
    entropy_reg_weight  = 0.01,
    reg_l2_prior_weight = 1e-3,
    proto_norm_weight   = 1e-3,
    contrastive         = True,
    contrastive_weight  = 0.1,
    contrastive_temp    = 0.1,
    vicreg              = True,
    vicreg_weight       = 0.05,
    barlow              = True,
    barlow_weight       = 0.005,
    triplet             = False,
    triplet_weight      = 0.1,

    # Curriculum (per stage)
    neg_prob_s1         = 0.40,
    neg_prob_s2         = 0.45,
    neg_prob_s3         = 0.50,
    hard_neg_ratio_s1   = 0.0,
    hard_neg_ratio_s2   = 0.30,
    hard_neg_ratio_s3   = 0.50,

    # Source-balanced batch composition (sums to batch_size)
    source_mix          = {'vizwiz_base': 4, 'vizwiz_novel': 2, 'hots': 5, 'insdet': 5},

    # Augmentation
    augment             = True,
    augment_strength    = 1.0,
    multi_scale_s2      = True,
    multi_scale_s3      = True,
    multi_scale_sizes   = (192, 224, 256),

    # Validation TTA
    val_use_tta         = True,
    val_tta_sizes       = (224, 288),

    # Checkpoints
    save_stage_completion = True,
    keep_last_n           = 6,

    # Resume — True (auto from last.pt), False (fresh), or a filename
    resume              = True,

    seed                = 42,
)


### Run training

Single call. Auto-resumes from `last.pt` if it exists. To start fresh, set `resume=False` or delete `MODEL_DIR/last.pt`.


In [ ]:
train(**TRAIN_KWARGS)


### Resume from a specific checkpoint (optional)

Examples:
```python
train(**{**TRAIN_KWARGS, 'resume': resume_from('stage1_complete.pt')})       # start Stage 2 fresh
train(**{**TRAIN_KWARGS, 'resume': resume_from('stage2_complete.pt')})       # start Stage 3 fold 0
train(**{**TRAIN_KWARGS, 'resume': resume_from('ckpt_s3_epoch012_fold1.pt')}) # resume mid-Stage-3
train(**{**TRAIN_KWARGS, 'resume': resume_from('best.pt')})                   # fork from best-by-metric
```


In [ ]:
# Example (commented out — copy-paste a filename to use):
# train(**{**TRAIN_KWARGS, 'resume': resume_from('stage2_complete.pt')})


---
## Evaluate on the held-out test split

Runs the kitchen-sink metric set (mAP@0.5 / mAP@[0.5:0.95] / per-IoU AP / IoU / containment / presence / collapse diagnostics / per-source breakdown) with two-scale TTA. Output JSON: `analysis/test_report.json`.


In [ ]:
evaluate_run(
    checkpoint = f'{MODEL_DIR}/best.pt',  # or 'stage3_complete.pt' / 'last.pt'
    manifest   = MANIFEST,
    split      = 'test',
    data_root  = DATA_ROOT,
    episodes   = 600,
    batch_size = 8,
    num_workers= 2,
    neg_prob   = 0.5,
    report     = f'{ANALYSIS_DIR}/test_report.json',
    use_tta    = True,
)


---
## Generate the offline report (PNGs)

Reads all per-epoch / per-fold / aggregate JSONs under `ANALYSIS_DIR` and writes plots into `ANALYSIS_DIR/plots/`.


In [ ]:
plot_all_from_jsons(ANALYSIS_DIR)
